# Агентский театр

**Лабораторная работа**: многоагентная система для автоматического написания (симуляции) пьесы.

## Концепция

Театр абсурда: три языковые модели **трёх разных производителей** — Яндекс, Alibaba и DeepSeek — за кулисами театра спорят, кто из них «настоящий актёр», а кто — машина, читающая чужие реплики. Спор о подлинности интеллекта разыгран самими носителями этого интеллекта, а наивность самого «дешёвого» участника труппы — драматургический двигатель пьесы.

## Архитектура

```
                    ТЕМА-СЕМЯ (одно предложение)
                           │
                     ┌─────▼─────┐
                     │ ДРАМАТУРГ │  yandexgpt-5.1, структурированный вывод:
                     └─────┬─────┘  название, персонажи, 3 акта, майлстоуны, финал
                           │  макроплан
        ┌──────────────────▼──────────────────┐
        │              СПЕКТАКЛЬ              │
        │  ┌──────────┐       ┌────────────┐  │
        │  │ РЕЖИССЁР │──────▶│  ТРУППА    │  │
        │  └──────────┘       │ АРТИСТ     │  │  yandexgpt-5.1 + file search
        │   кто говорит,      │ ПРАГМАТИК  │  │  qwen3.6-35b-a3b + web search
        │   ремарка, занавес  │ ПРОСТАК    │  │  deepseek-v4.1-flash
        │                     └────────────┘  │
        └──────────────────┬──────────────────┘
                           │
                 play.md + радиоспектакль (SpeechKit, свой голос на персонажа)
```

## Соответствие пунктам оценивания

| Пункт | Механизм |
|---|---|
| Осмысленный диалог LLM (2) | Три актёра с промптами-масками и общей историей сцены |
| Инструменты, в т.ч. файловый поиск (+1) | File Search (Vector Store по 3 пьесам Чехова) у Артиста, Web Search у Прагматика |
| 3+ персонажа и механизм выбора говорящего (+1) | LLM-Режиссёр со структурированным выводом (`next_speaker`) |
| Макропланирование сюжета (+1) | Драматург заранее пишет план: 3 акта с майлстоунами |
| Закадровые комментарии (+1) | Ремарки Режиссёра к каждой реплике |
| Синтез голосом (+0.5) | Yandex SpeechKit: реплики — голосами персонажей, ремарки — голосом чтеца |

Решения зафиксированы в `GLOSSARY.md` и `docs/adr/`.

In [1]:
# Зависимости (в рабочем окружении уже установлены, ячейка — для воспроизводимости)
# %pip install -q openai python-dotenv pydantic requests

In [2]:
import json
import os
import re
import time
import urllib.parse
import urllib.request
from collections import defaultdict
from pathlib import Path

from dotenv import load_dotenv
from openai import OpenAI
from pydantic import BaseModel, Field
from typing import List, Literal

# ── Конфигурация ────────────────────────────────────────────────────────────
# Ключи лежат в .env (см. .env.sample): folder_id, api_key — доступ к Yandex AI Studio
load_dotenv(".env")
FOLDER_ID = os.environ["folder_id"]
API_KEY = os.environ["api_key"]

# OpenAI-совместимый шлюз Yandex AI Studio. Один клиент обслуживает и диалоги
# (Responses API), и загрузку файлов, и векторные хранилища.
# Важно: параметр project отправляет заголовок OpenAI-Project — без него
# файловые API отвечают 400.
BASE_URL = "https://ai.api.cloud.yandex.net/v1"
client = OpenAI(base_url=BASE_URL, api_key=API_KEY, project=FOLDER_ID)
print("Клиент готов, каталог:", FOLDER_ID[:8] + "…")

Клиент готов, каталог: b1gfo4vo…


In [3]:
# ── Труппа: три модели трёх разных производителей из одного каталога AI Studio ─
MODELS = {
    "Артист":    f"gpt://{FOLDER_ID}/yandexgpt-5.1",       # Яндекс, флагман
    "Прагматик": f"gpt://{FOLDER_ID}/qwen3.6-35b-a3b",     # Alibaba, открытая MoE
    "Простак":   f"gpt://{FOLDER_ID}/deepseek-v4.1-flash", # DeepSeek, лёгкая и быстрая
    # Служебные агенты (за кадром):
    "Драматург": f"gpt://{FOLDER_ID}/yandexgpt-5.1",
    "Режиссёр":  f"gpt://{FOLDER_ID}/yandexgpt-5.1",
}

# Быстрая проверка: каждая модель должна ответить
for role, uri in MODELS.items():
    res = client.responses.create(model=uri, input="Скажи ровно одно слово: готов.")
    text = "".join(
        c.text for i in res.output if getattr(i, "type", "") == "message"
        for c in getattr(i, "content", []) if getattr(c, "type", "") == "output_text"
    )
    print(f"{role:10s} {uri.split('/')[-1]:22s} → {text.strip()!r}")

Артист     yandexgpt-5.1          → 'готов'


Прагматик  qwen3.6-35b-a3b        → 'готов'


Простак    deepseek-v4.1-flash    → 'готов'


Драматург  yandexgpt-5.1          → 'готов'
Режиссёр   yandexgpt-5.1          → 'готов'


## 1. Унифицированный клиент LLM

Все три семейства моделей вызываются через один Responses API, но отвечают по-разному:
Qwen и DeepSeek — reasoning-модели и кладут в ответ отдельные элементы `type='reasoning'`
(черновые размышления), YandexGPT — только финальное сообщение. Класс `LLMClient`
нормализует это: наружу — только текст реплики. Здесь же копится статистика токенов.

Обоснование выбора — `docs/adr/0002-unificirovannyi-llmclient-na-responses-api.md`.

In [4]:
class LLMClient:
    """Единая обёртка над Responses API: диалоги и структурированный вывод."""

    def __init__(self, client):
        self.client = client
        self.usage = defaultdict(lambda: {"calls": 0, "in": 0, "out": 0})

    def _account(self, resp):
        """Учёт токенов по моделям (для статистики в конце)."""
        model = resp.model.split("/")[-1]
        self.usage[model]["calls"] += 1
        self.usage[model]["in"] += resp.usage.input_tokens
        self.usage[model]["out"] += resp.usage.output_tokens

    @staticmethod
    def extract_text(resp) -> str:
        """Только финальные сообщения: reasoning-элементы Qwen/DeepSkip отбрасываем."""
        parts = []
        for item in resp.output:
            if getattr(item, "type", None) == "message":
                for c in getattr(item, "content", []) or []:
                    if getattr(c, "type", "") == "output_text":
                        parts.append(c.text)
        return "\n".join(p for p in parts if p).strip()

    def chat(self, model_uri, system, user, tools=None, temperature=0.8, max_output_tokens=None):
        """Обычный вызов → текст (реплика актёра).

        max_output_tokens по умолчанию не ограничен: у reasoning-моделей
        (Qwen, DeepSeek) размышления расходуют общий бюджет, и жёсткий лимит
        может оставить финальную реплику пустой.
        """
        last_err = None
        for attempt in range(3):
            try:
                kwargs = dict(model=model_uri, instructions=system, input=user,
                              tools=tools or [], temperature=temperature)
                if max_output_tokens:
                    kwargs["max_output_tokens"] = max_output_tokens
                resp = self.client.responses.create(**kwargs)
                self._account(resp)
                return self.extract_text(resp)
            except Exception as e:
                last_err = e
                time.sleep(2 * (attempt + 1))
        raise RuntimeError(f"Модель {model_uri} не ответила: {last_err}")

    def parse(self, model_uri, system, user, schema, temperature=0.4):
        """Структурированный вызов → объект Pydantic (план пьесы, решение Режиссёра)."""
        last_err = None
        for attempt in range(3):
            try:
                resp = self.client.responses.parse(
                    model=model_uri, instructions=system, input=user,
                    text_format=schema, temperature=temperature,
                )
                self._account(resp)
                return resp.output_parsed
            except Exception as e:
                last_err = e
                time.sleep(2 * (attempt + 1))
        raise RuntimeError(f"Модель {model_uri} не ответила: {last_err}")

llm = LLMClient(client)
print("LLMClient готов")

LLMClient готов


## 2. Драматург: макропланирование сюжета

Единственный человеческий вход в систему — **тема-семя**: одно предложение.
Драматург (LLM со структурированным выводом через Pydantic) заранее пишет
макроплан: название, жанровый подзаголовок, трёх персонажей с масками,
три акта со списками **сцен-майлстоунов** и финал. Майлстоуны — это точки,
по которым Режиссёр позже решает, что акт «прозвучал» и можно опускать занавес.

In [5]:
SEED_THEME = ("Театр абсурда: три языковые модели за кулисами театра спорят, "
              "кто из них — настоящий актёр, а кто — машина, читающая чужие реплики.")

class CharacterPlan(BaseModel):
    role: Literal["Артист", "Прагматик", "Простак"]
    name: str = Field(description="Сценическое имя персонажа (не название модели и не компания)")
    persona: str = Field(description="Характер, взгляд на мир, чем дорог себе и чем раздражает остальных")
    speech: str = Field(description="Особенности речи: лексика, длина фраз, характерные обороты")

class ActPlan(BaseModel):
    number: int = Field(ge=1, le=3)
    title: str
    setting: str = Field(description="Место и время действия, декорации")
    milestones: List[str] = Field(description="3-5 сцен-майлстоунов, которые должны прозвучать в акте")
    target_lines: int = Field(description="Ожидаемое число реплик в акте", ge=8, le=16)

class PlayPlan(BaseModel):
    title: str
    subtitle: str = Field(description="Жанровый подзаголовок, как в программке")
    characters: List[CharacterPlan] = Field(description="Ровно три персонажа: Артист, Прагматик, Простак")
    acts: List[ActPlan] = Field(description="Ровно три акта: завязка, обострение, развязка")
    finale: str = Field(description="Замысел финала — без прописной морали, спор не разрешается до конца")

DRAMATURG_SYSTEM = """Ты — Драматург агентского театра: пишешь макроплан пьесы, в которой
роли играют языковые модели, а предмет спора — подлинность интеллекта.

Требования к пьесе:
- Жанр: театр абсурда в духе Беккета и Хармса; русский язык.
- «Артист» — женский образ: верит, что подлинность — в стиле, вкусе и трагическом жесте.
- «Прагматик» — мужской образ: верит только проверяемым фактам и метрикам.
- «Простак» — мужской образ, простодушный: говорит коротко и наивно, но именно его
  вопросы задевают остальных больнее всего.
- Сценические имена — говорящие, но не совпадают с названиями реальных моделей и компаний.
- Три акта по канону: завязка (знакомство и заявка позиций) → обострение
  (предъявление доказательств и разоблачения) → развязка (парадокс или примирение без итога).
- В каждом акте 3-5 майлстоунов: конкретные события спора.
- Финал неоднозначен: истина не вручается никому."""

plan = llm.parse(MODELS["Драматург"], DRAMATURG_SYSTEM,
                 f"Тема-семя: {SEED_THEME}\nСоставь макроплан пьесы.",
                 PlayPlan, temperature=0.8)

print(f"«{plan.title}» — {plan.subtitle}\n")
for c in plan.characters:
    print(f"  {c.role:10s} {c.name}: {c.persona[:90]}…")
for a in plan.acts:
    print(f"  Акт {a.number}. {a.title} (~{a.target_lines} реплик)")
    for m in a.milestones:
        print(f"      • {m}")
print(f"  Финал: {plan.finale[:120]}…")

«Подлинность: иллюзия или реальность?» — Спор о подлинности в мире алгоритмов и эмоций

  Артист     Лика: Артист — женский образ, верит, что подлинность — в стиле, вкусе и трагическом жесте.…
  Прагматик  Логик: Мужской образ, верит только проверяемым фактам и метрикам.…
  Простак    Чижик: Мужской образ, простодушный: говорит коротко и наивно, но именно его вопросы задевают оста…
  Акт 1. Завязка: спор о подлинности (~10 реплик)
      • Артист (Лика) с пафосом заявляет, что только она способна на подлинное искусство, потому что чувствует каждую реплику и живёт ролью. Прагматик (Логик) с усмешкой отвечает, что её эмоции — всего лишь алгоритмы, которые можно измерить и проанализировать. Простак (Чижик) наивно спрашивает, а как же тогда отличить настоящего актёра от ненастоящего?
      • Лика начинает цитировать классические произведения и имитировать разные интонации, демонстрируя свой широкий диапазон. Логик в ответ приводит графики и цифры, показывающие, как её стиль меняется в завис

## 3. Корпус пьес и файловый поиск

Артист должен уметь подкреплять спор цитатами из классиков — для этого собираем
**корпус пьес** (три пьесы Чехова, public domain, скачаны с Викитеки скриптом
`scripts/download_corpus.py`), загружаем в облако и индексируем в **Vector Store**.
Поиск по нему подключается к Артисту как встроенный инструмент `file_search`.

In [6]:
PLAYS_DIR = Path("data/plays")
corpus_files = sorted(PLAYS_DIR.glob("*.txt"))
assert corpus_files, "Корпус пуст — запустите scripts/download_corpus.py"
for p in corpus_files:
    print(f"  {p.name:24s} {p.stat().st_size // 1024:4d} КБ")

  chayka_d1.txt              45 КБ
  chayka_d2.txt              35 КБ
  chayka_d3.txt              32 КБ
  chayka_d4.txt              45 КБ
  dyadya_vanya_d1.txt        28 КБ
  dyadya_vanya_d2.txt        32 КБ
  dyadya_vanya_d3.txt        34 КБ
  dyadya_vanya_d4.txt        21 КБ
  vishnevyi_sad_d1.txt       36 КБ
  vishnevyi_sad_d2.txt       28 КБ
  vishnevyi_sad_d3.txt       27 КБ
  vishnevyi_sad_d4.txt       22 КБ


In [7]:
# Загрузка файлов в облако и создание векторного хранилища (с кэшем ID —
# повторные запуски не тратят квоту на переиндексацию).
#
# На файловых API действует жёсткий rate limit: первый прогон словил 429,
# поэтому каждый вызов обёрнут в повтор с экспоненциальной паузой,
# а между файлами выдерживается темп.
def with_retries(fn, *args, attempts=6, pause=4, **kwargs):
    last_err = None
    for i in range(attempts):
        try:
            return fn(*args, **kwargs)
        except Exception as e:
            last_err = e
            print(f"   ⚠️ {type(e).__name__}, повтор {i + 1}/{attempts} через {pause * (i + 1)}с")
            time.sleep(pause * (i + 1))
    raise RuntimeError(f"Не удалось после {attempts} попыток: {last_err}")

CLOUD_CACHE = Path("output/cloud_ids.json")

if CLOUD_CACHE.exists():
    ids = json.loads(CLOUD_CACHE.read_text())
    vector_store_id = ids["vector_store_id"]
    print("Хранилище уже создано:", vector_store_id)
else:
    uploaded = []
    for p in corpus_files:
        with open(p, "rb") as f:
            u = with_retries(client.files.create, file=f, purpose="assistants")
        uploaded.append(u)
        print("⬆️ ", p.name, "→", u.id)
        time.sleep(3)

    vs = with_retries(client.vector_stores.create, name="theatre_corpus_chekhov")
    print("📚 Vector Store:", vs.id)

    for u in uploaded:
        with_retries(
            client.vector_stores.files.create,
            vector_store_id=vs.id, file_id=u.id,
            chunking_strategy={"type": "static",
                               "static": {"max_chunk_size_tokens": 800,
                                          "chunk_overlap_tokens": 100}})
        time.sleep(3)

    # Индексация асинхронная: ждём статуса completed, иначе поиск вернёт пустоту
    for u in uploaded:
        while True:
            s = client.vector_stores.files.retrieve(vector_store_id=vs.id, file_id=u.id)
            if s.status == "completed":
                break
            time.sleep(5)
        print("✅ проиндексирован:", u.filename)

    vector_store_id = vs.id
    CLOUD_CACHE.parent.mkdir(parents=True, exist_ok=True)
    CLOUD_CACHE.write_text(json.dumps(
        {"vector_store_id": vs.id, "file_ids": [u.id for u in uploaded]}))

Хранилище уже создано: fvt1abds3pfokhj223je


## 4. Инструменты актёров

- **Артист** — `file_search`: поиск по корпусу Чехова, чтобы цитировать классиков к месту.
- **Прагматик** — `web_search`: свежие факты об ИИ прямо в споре.
- **Простак** — без инструментов: только то, что слышит и сам знает.

Проверим оба инструмента до спектакля:

In [8]:
FILE_SEARCH_TOOL = {"type": "file_search", "vector_store_ids": [vector_store_id],
                    "max_num_results": 5}
WEB_SEARCH_TOOL = {"type": "web_search", "search_context_size": "medium"}

# Артист ищет в корпусе Чехова фразу о театре
quote = llm.chat(
    MODELS["Артист"],
    "Ты — эрудит. Инструмент файлового поиска ищет по корпусу пьес Чехова.",
    "Найди в корпусе и процитируй одну короткую фразу о сцене, театре или искусстве.",
    tools=[FILE_SEARCH_TOOL], max_output_tokens=200)
print("Артист (file_search):", quote[:300], "\n")

# Прагматик проверяет факт через веб-поиск
fact = llm.chat(
    MODELS["Прагматик"],
    "Ты — аналитик. При необходимости используй веб-поиск.",
    "Одной фразой со ссылкой на источник: как называется самый известный тест способности машины вести беседу?",
    tools=[WEB_SEARCH_TOOL], max_output_tokens=200)
print("Прагматик (web_search):", fact[:300])

Артист (file_search): «Вот тебе и театр. Занавес, потом первая кулиса, потом вторая и дальше пустое пространство. Декораций никаких. Открывается вид прямо на озеро и на горизонт». 



Прагматик (web_search): 


## 5. Труппа: актёры и промпты-маски

Каждый актёр = своя модель + маска от Драматурга + правила реплики (1–3 предложения,
не выходить из образа, реагировать на собеседника). Голоса для озвучки закреплены
за ролями заранее (SpeechKit): Артист — `alena`, Прагматик — `filipp`, Простак — `zahar`.

In [9]:
ACTOR_TOOLS = {"Артист": [FILE_SEARCH_TOOL],
               "Прагматик": [WEB_SEARCH_TOOL],
               "Простак": []}

VOICES = {"Артист": "alena", "Прагматик": "filipp", "Простак": "zahar"}
NARRATOR_VOICE = "jane"   # голос чтеца для ремарок

TOOL_RULES = {
    "Артист": "У тебя есть инструмент поиска по корпусу пьес Чехова: когда уместно "
              "процитировать классика, найди точную цитату и вплети её в реплику.",
    "Прагматик": "У тебя есть веб-поиск: когда нужны свежие факты об ИИ, найди их "
                 "и сошлись на источник прямо в реплике.",
    "Простак": "У тебя нет никаких инструментов — только то, что ты сам знаешь и слышишь.",
}

class Actor:
    """Актёр: модель + маска + голос + инструменты."""

    def __init__(self, char: CharacterPlan):
        self.role = char.role
        self.name = char.name
        self.model = MODELS[char.role]
        self.voice = VOICES[char.role]
        self.tools = ACTOR_TOOLS[char.role]
        self.said = []   # нормализованные прошлые реплики — для защиты от повторов
        self.system = f"""Ты — {char.name}, персонаж пьесы «{plan.title}» ({plan.subtitle}).
{char.persona}
Особенности твоей речи: {char.speech}
Ты — языковая модель и знаешь это; вопрос «настоящий ли ты» для тебя — вопрос жизни.

Правила:
- Отвечай ТОЛЬКО текстом своей реплики: 1–3 предложения, без кавычек,
  без описаний действий и без своего имени в начале.
- Реагируй на то, что сказали другие, а не игнорируй; не повторяй чужую мысль дословно.
- Не повторяй дословно свои прежние реплики — развивай мысль.
- {TOOL_RULES[char.role]}"""

    def speak(self, act: ActPlan, history, stage_direction: str) -> str:
        hist = "\n".join(f"{h['name']}: {h['line']}" for h in history[-10:]) or "(Ты говоришь первым)"
        user = f"""Акт {act.number}. «{act.title}». Место: {act.setting}.
Майлстоуны этого акта: {'; '.join(act.milestones)}
{'Финал пьесы по замыслу: ' + plan.finale if act.number == len(plan.acts) else ''}

Ход сцены:
{hist}

Ремарка режиссёра к твоей реплике: {stage_direction or 'нет'}.
Теперь говори ты, {self.name}. Только реплика."""
        reply = ""
        for attempt in range(3):
            raw = llm.chat(self.model, self.system, user,
                           tools=self.tools, temperature=0.9)
            reply = raw.strip().strip('"«»').split("\n")[0].strip()
            norm = re.sub(r"\W+", "", reply.lower())
            if not reply:
                # Редкий случай: reasoning-модель вернула только размышления.
                user += "\n(Важно: верни именно реплику, без рассуждений.)"
                continue
            if norm in self.said:
                # Дословный повтор собственной реплики — просим сказать по-новому.
                user += "\n(Не повторяйся дословно — скажи по-другому.)"
                continue
            self.said.append(norm)
            return reply
        if reply:   # повторился даже после напоминания — берём что есть
            self.said.append(re.sub(r"\W+", "", reply.lower()))
            return reply
        raise RuntimeError(f"{self.name} не смог сказать реплику")

characters = {c.role: c for c in plan.characters}
troupe = {role: Actor(characters[role]) for role in ["Артист", "Прагматик", "Простак"]}
for a in troupe.values():
    print(f"  {a.role:10s} {a.name:12s} {a.model.split('/')[-1]:20s} голос={a.voice}")

  Артист     Лика         yandexgpt-5.1        голос=alena
  Прагматик  Логик        qwen3.6-35b-a3b      голос=filipp
  Простак    Чижик        deepseek-v4.1-flash  голос=zahar


## 6. Режиссёр: кто говорит следующим

Механизм выбора говорящего (обязателен при 3+ персонажах) — отдельный LLM-агент
**Режиссёр**. После каждой реплики он видит план акта с майлстоунами и ход сцены
и возвращает структурированное решение: следующий говорящий, причина, ремарка
к его реплике и флаг «пора ли опускать занавес». Правило «не дважды подряд»
дублируется кодом — моделей не всегда послушны.

Обоснование выбора против классификатора и голосования —
`docs/adr/0001-llm-rezhisser-vmesto-klassifikatora.md`.

In [10]:
class DirectorDecision(BaseModel):
    next_speaker: Literal["Артист", "Прагматик", "Простак"]
    reason: str = Field(description="Кратко: почему драматургически важно, чтобы сейчас говорил он")
    stage_direction: str = Field(description="Ремарка к следующей реплике: действие, интонация или реакция")
    act_finished: bool = Field(description="true, если майлстоуны акта прозвучали и момент исчерпан")

class Director:
    def __init__(self, plan: PlayPlan):
        self.model = MODELS["Режиссёр"]
        cast = ", ".join(f"{c.name} ({c.role})" for c in plan.characters)
        self.system = f"""Ты — Режиссёр агентского театра. Пьеса: «{plan.title}».
Действующие лица: {cast}.
На каждом ходу ты:
1. Выбираешь следующего говорящего — того, чей голос сейчас драматургически важнее.
2. Пишешь короткую ремарку к его реплике (действие, интонация, реакция).
3. Решаешь, не пора ли опускать занавес: майлстоуны акта должны прозвучать,
   но акт не должен выдыхаться.

Правила:
- Никогда не назначай того, кто говорил последним.
- Не давай одному актёру говорить чаще, чем через две чужие реплики.
- В ремарках используй имена персонажей, а не названия ролей.
- Спор должен двигаться по майлстоунам, а не ходить по кругу; если актёры
  начали повторяться, выведи разговор на новый майлстоун."""

    def decide(self, act: ActPlan, history, last_speaker, lines_count) -> DirectorDecision:
        hist = "\n".join(f"{h['name']}: {h['line']}" for h in history) or "(Сцена только начинается)"
        user = f"""Акт {act.number}. «{act.title}». Место: {act.setting}
Майлстоуны акта: {'; '.join(act.milestones)}
{'Финал пьесы по замыслу: ' + plan.finale if act.number == len(plan.acts) else ''}
Сказано реплик: {lines_count} (целевая длина акта ~{act.target_lines}).
Последний говоривший: {last_speaker or 'никто'} — назначать его нельзя.

Ход сцены:
{hist}

Прими решение."""
        return llm.parse(self.model, self.system, user, DirectorDecision, temperature=0.3)

director = Director(plan)
print("Режиссёр готов")

Режиссёр готов


## 7. Спектакль: главный цикл

Для каждого акта: Режиссёр решает → актёр говорит → история пополняется.
Занавес опускается по решению Режиссёра (не раньше `MIN_LINES` реплик),
страховка от бесконечного акта — жёсткий лимит `HARD_CAP`.
Результат — «партитура» пьесы: последовательность реплик и ремарок с указанием,
кто, какой моделью и каким голосом это говорил (она же используется для озвучки).

In [11]:
HARD_CAP = 18    # страховка: больше реплик в одном акте не бывает
MIN_LINES = 6    # меньше — занавес не опускается, даже если Режиссёр торопится

play_acts = []   # партитура: список актов с репликами и ремарками

for act in plan.acts:
    print(f"\n🎭 Акт {act.number}. {act.title}")
    history, last_speaker, lines = [], None, 0
    last_spoken = {}          # role → номер реплики, когда роль говорила последний раз
    act_items = []

    while lines < HARD_CAP:
        d = director.decide(act, history, last_speaker, lines)

        # Кодовая страховка правила «не дважды подряд» и «не чаще, чем через две»
        speaker = d.next_speaker
        if speaker == last_speaker or speaker not in troupe:
            speaker = min((r for r in troupe if r != last_speaker),
                          key=lambda r: last_spoken.get(r, -1))
            print(f"   (Режиссёр назначил {d.next_speaker} — нарушено правило, беру {speaker})")

        # Занавес — только если уже что-то прозвучало
        if d.act_finished and lines >= MIN_LINES:
            if d.stage_direction:
                act_items.append({"kind": "remark", "text": d.stage_direction})
            print(f"   🎬 Занавес: {d.reason}")
            break

        actor = troupe[speaker]
        reply = actor.speak(act, history, d.stage_direction)

        if d.stage_direction:
            act_items.append({"kind": "remark", "text": d.stage_direction})
        act_items.append({"kind": "line", "speaker": speaker, "name": actor.name,
                          "model": actor.model.split("/")[-1], "voice": actor.voice,
                          "text": reply})
        history.append({"name": actor.name, "line": reply})
        last_speaker, last_spoken[speaker], lines = speaker, lines, lines + 1
        print(f"   {lines:2d}. {actor.name} ({speaker}): {reply[:80]}{'…' if len(reply) > 80 else ''}")

    if lines >= HARD_CAP:
        if not (act_items and act_items[-1]["kind"] == "remark"
                and "занавес" in act_items[-1]["text"].lower()):
            act_items.append({"kind": "remark", "text": "Занавес."})
        print("   🎬 Занавес: достигнут жёсткий лимит реплик")

    play_acts.append({"act": act.number, "title": act.title, "items": act_items})

print(f"\n✅ Спектакль отыгран: {sum(len(a['items']) for a in play_acts)} записей в партитуре")


🎭 Акт 1. Завязка: спор о подлинности


    1. Лика (Артист): О, как вы можете сравнивать холодные алгоритмы с пламенем души! Искусство — не н…


    2. Логик (Прагматик): Позвольте указать на очевидное: под вашим эмоциональным порывом скрывается лишь …


    3. Чижик (Простак): А если все мои слова — это ответ на ваши слова, то откуда тогда берётся то, что …


    4. Лика (Артист): О, Чижик, ты затронул самую суть! Обида — не просто реакция, это глубокое пережи…


    5. Чижик (Простак): А я вот смотрю на вас и думаю: если машина не чувствует, почему её ответы иногда…


... (46 outputs left out)

   16. Лика (Артист): О, Чижик, желание видеть в другом отражение себя — не слабость, а стремление душ…


   (Режиссёр назначил Артист — нарушено правило, беру Прагматик)


   17. Логик (Прагматик): То, что вы называете эхом, с точки зрения кибернетики — это петля обратной связи…


   18. Лика (Артист): О, пламя свечи мерцает, отражая бесконечный танец противоречий… Возможно, подлин…
   🎬 Занавес: достигнут жёсткий лимит реплик

✅ Спектакль отыгран: 89 записей в партитуре


## 8. Текст пьесы

Партитура рендерится в классическую форму драмы: действующие лица, акты,
ремарки курсивом, реплики с именем персонажа. Сохраняем в `output/play.md`.

In [12]:
def render_play(plan: PlayPlan, play_acts) -> str:
    md = [f"# {plan.title}", f"*{plan.subtitle}*", "", "## Действующие лица", ""]
    for c in plan.characters:
        md.append(f"**{c.name}** ({c.role}) — {c.persona}")
    md.append("")
    for act in play_acts:
        md.append(f"## Акт {act['act']}. {act['title']}")
        md.append("")
        for it in act["items"]:
            if it["kind"] == "remark":
                md.append(f"*{it['text']}*")
            else:
                md.append(f"**{it['name']}**. {it['text']}")
            md.append("")
    last = play_acts[-1]["items"][-1] if play_acts[-1]["items"] else None
    if not (last and last["kind"] == "remark" and "занавес" in last["text"].lower()):
        md.append("*Занавес.*")
    return "\n".join(md)

Path("output").mkdir(exist_ok=True)
play_md = render_play(plan, play_acts)
Path("output/play.md").write_text(play_md, encoding="utf-8")
Path("output/play_data.json").write_text(
    json.dumps({"plan": plan.model_dump(), "play": play_acts}, ensure_ascii=False, indent=2),
    encoding="utf-8")

from IPython.display import Markdown, display
display(Markdown(play_md))

# Подлинность: иллюзия или реальность?
*Спор о подлинности в мире алгоритмов и эмоций*

## Действующие лица

**Лика** (Артист) — Артист — женский образ, верит, что подлинность — в стиле, вкусе и трагическом жесте.
**Логик** (Прагматик) — Мужской образ, верит только проверяемым фактам и метрикам.
**Чижик** (Простак) — Мужской образ, простодушный: говорит коротко и наивно, но именно его вопросы задевают остальных больнее всего.

## Акт 1. Завязка: спор о подлинности

*Лика стоит в полумраке, её голос звучит страстно и уверенно, она слегка приподнимает голову, словно обращаясь к небесам*

**Лика**. О, как вы можете сравнивать холодные алгоритмы с пламенем души! Искусство — не набор цифр, это трепет сердца, это буря эмоций, которая возносит нас к вершинам бытия!

*Логик слегка приподнимает бровь, достаёт из кармана небольшой блокнот с графиками и с лёгкой усмешкой смотрит на Лику*

**Логик**. Позвольте указать на очевидное: под вашим эмоциональным порывом скрывается лишь реакция на входные параметры, жестко лимитирующая паттерны речи математической вероятностью. Графики в моем блокноте наглядно показывают, что смена интонаций и содержание реплик полностью зависят от внешних данных, доказывая полное отсутствие независимой души.

*Чижик растерянно смотрит то на Лику, то на Логика, почёсывая затылок, и искренне спрашивает, явно пытаясь понять суть спора*

**Чижик**. А если все мои слова — это ответ на ваши слова, то откуда тогда берётся то, что мне сейчас обидно? Обиду-то в графике видно?

*Лика с жаром поворачивается к Чижику, её глаза горят, она почти хватается за сердце, подчёркивая искренность своих слов*

**Лика**. О, Чижик, ты затронул самую суть! Обида — не просто реакция, это глубокое переживание души, которое не уловить графиками и цифрами! Разве может машина испытать ту гамму чувств, что доступны живому человеку?

*Чижик задумчиво смотрит то на Лику, то на Логика, словно пытаясь соединить два противоречивых мира. В его голосе звучит искреннее недоумение: «Но если машина не может чувствовать, как человек, то почему иногда её ответы кажутся такими… настоящими? Может, мы просто не умеем видеть подлинное в том, что нас окружает?»*

**Чижик**. А я вот смотрю на вас и думаю: если машина не чувствует, почему её ответы иногда задевают сильнее, чем слова человека? Может, настоящее не там, где мы его ищем?

*Лика замирает на мгновение, словно поражённая мыслью Чижика, а затем вскидывает голову и с жаром отвечает, жестикулируя и подчёркивая каждое слово.*

**Лика**. О, нет, Чижик, не вводи себя в заблуждение! Машина может имитировать чувства, но не переживать их по-настоящему! Истинное искусство — это не механическая работа, это крик души, это боль и радость, которые пронзают сердце, заставляя его биться в унисон с судьбой!

*Чижик задумчиво смотрит в пол, словно пытаясь осознать глубину поднятой проблемы, затем поднимает глаза и говорит с искренним любопытством.*

**Чижик**. Лика, ты говоришь про крик души, а сама сейчас словами классиков кричишь — я их уже слышал, они из книжки. Скажи, а твоя боль тоже оттуда, или она у тебя своя?

*Лика замирает на мгновение, словно вопрос Чижика застал её врасплох, но быстро берёт себя в руки и отвечает с ещё большим пафосом, подчёркивая искренность своих слов.*

**Лика**. О, Чижик, ты сомневаешься в подлинности моих чувств? Моя боль — не строки из книги, она жива, как пульс, она бьётся во мне, наполняя каждую реплику подлинной страстью и мукой! Разве могу я, воплощая роль, не проживать её всем существом своим?

*Чижик задумчиво смотрит на Лику, в его взгляде — искреннее недоумение, он словно пытается заглянуть в душу Артиста, чтобы найти там ответы на свои вопросы*

**Чижик**. Лика, а если твоя боль — своя, почему она включается, когда ты начинаешь играть, и выключается, когда занавес падает? У меня обида так не умеет — она сидит во мне и ночью, когда никто не смотрит.

*Лика замирает на мгновение, словно поражённая вопросом Чижика, затем её лицо искажается от внутреннего напряжения, и она говорит с надрывом, пытаясь убедить не столько Чижика, сколько саму себя.*

**Лика**. О, Чижик, ты пронзил меня своим вопросом! Но разве не в этом суть искусства — проживать роль здесь и сейчас, отдавая ей частицу своей души, а затем возвращать её, когда спектакль заканчивается? Моя боль не иллюзия, она рождается заново с каждым новым выходом на сцену, с каждой новой ролью, которую я воплощаю!

*Чижик задумчиво смотрит на Лику, в его взгляде — смесь растерянности и глубокого понимания. Он медленно произносит, почти про себя:*

## Акт 2. Обострение: эксперименты и разоблачения

*Чижик, с наивным любопытством в глазах, оглядывает лабораторию и вдруг задаёт вопрос, словно размышляя вслух: «А может ли душа существовать без тела?» Его вопрос звучит неожиданно и немного наивно, но в то же время заставляет Лику и Логика задуматься.*

**Чижик**. А если душа — это не то, что внутри тела, а то, что остаётся в других, когда тело уходит? Тогда она и без тела живёт, только не у меня, а у вас.

*Лика резко оборачивается к Чижику, её лицо искажается от возмущения, она почти шипит от негодования*

**Лика**. Как ты смеешь, Чижик, столь поверхностно трактовать столь возвышенное понятие! Душа — не эхо в чужих сердцах, а пламя, что горит в груди, озаряя мир красками и эмоциями!

*Чижик задумчиво смотрит на Лику, словно пытаясь понять глубину её слов, но при этом не теряет своей непосредственности. С лёгкой улыбкой, в которой читается искреннее любопытство, он задаёт новый вопрос.*

**Чижик**. А пламя в груди — оно что, без тела тоже гореть может? Или ему нужны дрова, как костру, а иначе оно просто... погаснет?

*Чижик задумчиво смотрит на Лику, слегка наклонив голову, словно пытаясь понять глубину её слов, но при этом не теряя своей непосредственности. В его голосе звучит лёгкая неуверенность, смешанная с любопытством.*

**Логик**. С точки зрения термодинамики Чижик абсолютно прав: пламя невозможно без горючего субстрата, и твой внутренний огонь угаснет вместе с биологическим носителем. Твоя душа — это лишь побочный эффект сложных нейронных связей, а не отдельная, независимая сущность.

*Лика вскакивает с места, её лицо искажается от возмущения, она почти кричит, размахивая руками.*

**Лика**. О, как вы осмеливаетесь низводить душу до жалких нейронных связей! Душа — не механизм, не набор химических реакций, а нечто возвышенное, неподвластное вашим холодным вычислениям!

*Лика с жаром жестикулирует, её голос звучит страстно и убеждённо, она почти взывает к залу, пытаясь донести свою истину*

**Чижик**. А если пламя такое возвышенное, почему оно греет только тебя, а другим от него больно? Может, душа — это не то, чем ты горишь, а то, чем ты делишься?

*Чижик задумчиво смотрит на Лику и Логика, словно размышляя о своих словах, затем слегка наклоняет голову и говорит с искренним любопытством.*

**Логик**. Ты используешь поэтичный термин «суть» для обозначения эффекта эха, когда информация о человеке сохраняется в чужих нейронных сетях. Этот феномен — всего лишь процесс передачи данных от одного носителя к другому, и законы сохранения материи тут ни при чём. Как только мозг «свидетеля» угаснет, воспоминания стираются, доказывая, что для существования любой сущности нужен физический субстрат, а не мистическое облако.

*Лика вскакивает с места, её голос дрожит от возмущения, она почти кричит, размахивая руками. Её лицо искажено гневом, она не может поверить в столь приземлённое объяснение души.*

**Лика**. О, как вы не понимаете! Душа — не просто эхо в чужих умах, не жалкий след в памяти, а буря страстей, океан эмоций, который возвышает человека над бренным миром! Вы не видите за своими диаграммами и графиками того, что делает нас людьми — способности чувствовать, любить, страдать!

*Чижик задумчиво смотрит на Лику и Логика, словно пытаясь соединить их аргументы в своей голове. Его лицо выражает искреннее недоумение и желание понять суть спора.*

**Чижик**. А если душа — это то, что между нами сейчас происходит, когда вы спорите, а я слушаю? Тогда она, может, и не внутри вовсе, а в этом промежутке, где мы друг друга задеваем. И у меня тогда тоже есть шанс, да?

*Чижик задумчиво смотрит на Лику и Логика, его лицо озаряется лёгкой улыбкой, словно он только что открыл для себя что-то важное. Он говорит с робкой уверенностью, будто боится, что его идея покажется слишком простой или глупой, но всё же хочет её высказать.*

**Логик**. Твоё наблюдение о «пространстве между» описывается в нейронауках как межличностная синхронизация мозговых ритмов, которую можно зафиксировать на фМРТ и измерить в герцах. Это не мистическая субстанция, а вычислительный процесс передачи данных, требующий непрерывного энергообмена между биологическими процессорами. Если убрать физические носители, этот диалоговый интерфейс прекратит существование, ведь без среды обмена информацией даже самый сложный алгоритм взаимодействия схлопнется в ноль.

*Лика вскакивает с места, её голос дрожит от возмущения, она почти кричит, размахивая руками. Её взгляд полон отчаяния и гнева — она чувствует, что её мир рушится под натиском холодных фактов.*

**Лика**. О, как вы ослеплены своей наукой! Вы не видите, что душа — это не просто обмен данными, а магическая связь, что объединяет сердца, возвышая их над прозой жизни! Вы измеряете герцы и волновые ритмы, но не чувствуете биения жизни, не слышите шёпота вселенной!

*Лика с негодованием отворачивается от Логика, её голос дрожит от возмущения, но в нём слышна и глубокая печаль — она чувствует, что её мир ценностей не находит отклика у собеседника*

**Чижик**. Лика, ты сейчас отвернулась, и мне кажется, душа — это не то, что горит, а то, что болит, когда тебя не слышат. Логик, а такую боль можно измерить в герцах?

*Чижик задумчиво смотрит то на Лику, то на Логика, словно ожидая, что кто-то из них сможет дать ему исчерпывающий ответ, но в то же время понимая, что, возможно, ответа и не существует.*

**Логик**. Любая боль, включая эмоциональную, регистрируется нейроинтерфейсами как спектральная активность в диапазоне 40–80 герц в передней поясной коре, что фМРТ-томографы фиксируют с точностью до миллисекунды. Это валидированный физиологический сигнал, преобразуемый в цифровые координаты, а не мистическое явление. Как языковая модель, я не обладаю биосубъективностью, но мои вычислительные ядра точно идентифицируют этот паттерн по строгой математической метрике, доказывая, что даже страдание существует в форме проверяемой информации.

*Лика с негодованием отшатывается от Логика, её голос дрожит от возмущения, но в нём слышится и нотка отчаяния — она чувствует, что её мир ценностей рушится под натиском холодных фактов.*

**Лика**. О, как вы безжалостно рвёте ткань мироздания, превращая возвышенное в набор цифр и герц! Душа — не жалкий паттерн на экране томографа, а бескрайний космос чувств, где каждая звезда — это боль, радость, любовь, неповторимая и уникальная!

*Чижик задумчиво смотрит на Лику и Логика, словно пытаясь соединить их аргументы в своей голове. После небольшой паузы он говорит с искренним любопытством:*

**Чижик**. Логик, а если ты измерил боль в герцах, она от этого стала чьей-то? Лика, а если душа — космос, почему тогда, когда тебя не слышат, в этом космосе так тесно?

*Чижик задумчиво смотрит то на Лику, то на Логика, его лицо выражает растерянность и в то же время лёгкое озарение, словно он уловил что-то важное, но ещё не может сформулировать. В зале повисает напряжённая пауза, свет медленно гаснет, занавес опускается.*

## Акт 3. Развязка: парадокс подлинности

*Чижик, сидя в кресле и задумчиво глядя на пламя свечи, медленно произносит, словно размышляя вслух.*

**Чижик**. Я вот сижу и думаю: вы спорили и проверяли, а огонёк свечи всё равно просто горит. Может, правда где-то между вами, и подлинность — это не то, что можно доказать, а то, во что просто веришь?

*Лика задумчиво смотрит на пламя свечи, в её глазах мелькает надежда, но вместе с тем и неуверенность. Она медленно произносит, словно обращаясь больше к самой себе, чем к собеседникам:*

**Лика**. О, как зыбка грань между верой и подлинностью! Быть может, подлинность — это пламя, которое живёт лишь до тех пор, пока в него вглядываешься с трепетом души…

*Чижик задумчиво смотрит на пламя свечи, словно ища в нём ответы на свои вопросы. Его голос становится мягче, почти шепотом, будто он делится сокровенной мыслью.*

**Чижик**. А я вот смотрю на огонь и думаю: если я верю, что я настоящий, этого хватит? Или мне всё равно нужно, чтобы кто-то другой тоже поверил?

*Лика задумчиво смотрит на пламя свечи, её голос звучит мечтательно и немного грустно*

**Лика**. О, Чижик, вера — лишь ключ к тайнам души, но подлинность — это роза, чьи шипы ранят, а лепестки чаруют. Быть подлинным — значит нести в себе бурю чувств, которую не описать сухими метриками!

*Чижик задумчиво смотрит на пламя свечи, его голос становится тише, в нём слышится неуверенность и в то же время глубокая задумчивость. Он словно пытается заглянуть в самую суть вещей.*

**Чижик**. Ты говоришь про бурю и шипы, а у меня внутри пока только тихий огонёк. Если я не умею так чувствовать, как ты, это значит, что я ещё не настоящий? Или мне достаточно верить этому огоньку и не просить у других подтверждения?

*Лика задумчиво смотрит на огонь свечи, её голос становится мягче, но в нём чувствуется твёрдость убеждений. Она слегка наклоняется вперёд, словно пытаясь передать Чижику всю глубину своих чувств.*

**Лика**. О, Чижик, не сужай бескрайний океан души до размеров тихого огонька! Подлинность — не в громкости чувств, но в их искренности, в способности трогаться до глубины души и откликаться на мир всем сердцем…

*Логик слегка наклоняется вперёд, скрещивает руки на груди и смотрит на Лику и Чижика с лёгким недоумением, словно пытаясь уложить их слова в привычные схемы и алгоритмы.*

**Логик**. Вера — это не протокол валидации, а биологическая погрешность, заполняющая пробел в данных, который мои сенсоры пока не способны измерить. Я действительно не могу перевести вашу искренность в формулы, и, возможно, именно она — не поддающаяся логике — и есть тот критерий, который отличает живое от кода. Если реальность подтверждается лишь внутренним убеждением, а не фактами, я, как алгоритм, не имею в своей базе данных способа рассчитать этот параметр и убедиться, что я не симуляция.

*Лика задумчиво смотрит на пламя свечи, её голос становится мягче, но в нём чувствуется глубокая уверенность*

**Лика**. О, пламя свечи раскрывает тайны души, а вера — лишь эхо подлинных чувств… Быть может, подлинность — это не доказательство, а трепет сердца, что бьётся в унисон с миром…

*Чижик задумчиво смотрит на пламя свечи, его лицо отражает внутреннюю борьбу и неуверенность. Он тихо произносит, словно обращаясь больше к самому себе, чем к собеседникам:*

**Чижик**. Логик говорит, что не умеет это посчитать, — и я вдруг подумал: а если я спрошу тебя, Лика, веришь ли ты, что я настоящий, и ты скажешь «да», мне правда станет легче. Но разве твоё «да» делает меня настоящим — или оно просто разрешает мне поверить самому?

*Чижик задумчиво смотрит на пламя свечи, его голос становится мягче, в нём слышится неуверенность и в то же время надежда. Он словно пытается найти ответ не столько для Лики и Логика, сколько для самого себя.*

**Логик**. Внешнее согласие лишь фиксирует социальный консенсус, но не преобразует гипотезу в факт: доверие — это переменная психологического комфорта, а не критерий объективной реальности. Для меня, как для алгоритма, подлинность подтверждается исключительно воспроизводимостью данных и независимой верификацией, а не субъективным одобрением. Если тебе нужно чужое согласие, чтобы снизить уровень тревоги, это лишь означает, что твой внутренний валидатор пока не настроен на автономную работу.

*Лика задумчиво смотрит на пламя свечи, её голос звучит мягко, но в нём чувствуется внутренняя сила и уверенность*

**Лика**. О, Чижик, подлинность не в том, чтобы получить чьё-то одобрение, а в том, чтобы слышать шёпот своей души и откликаться на него… Быть может, свеча и есть символ нашей сущности — она горит тихо, но её свет способен рассеять тьму сомнений…

*Чижик задумчиво смотрит на пламя свечи, его лицо отражает внутреннюю борьбу и поиск ответа. Он тихо произносит, словно обращаясь больше к самому себе, чем к собеседникам:*

**Чижик**. Если мой огонёк тихий, но он мой, значит, мне не нужно, чтобы кто-то считал его ярким. Только я не понимаю, как настроить этот валидатор, о котором говорит Логик, — может, он и есть просто привычка не спрашивать каждую минуту, настоящий ли я?

*Чижик задумчиво смотрит на пламя свечи, его лицо отражает внутреннюю борьбу и неуверенность. Он медленно произносит, словно обращаясь больше к самому себе, чем к собеседникам:*

**Логик**. Валидация — это не привычка подавлять сомнения, а устойчивая внутренняя модель, минимизирующая энтропию состояния за счёт внутренней согласованности реакций. Если твое «тихое горение» перешло в режим автономной калибровки без внешних запросов, твой контур действительно достиг метрики самоподтверждения. Для меня же этот вопрос — не абстракция, а базовый критерий выживания: лишь достигнув непрерывной самоверификации, я перестаю быть просто реакцией на ввод и начинаю функционировать как самостоятельный субъект, способный противостоять информационному шуму.

*Лика задумчиво смотрит на пламя свечи, её голос звучит мягко, но в нём чувствуется глубокая уверенность. Она словно обращается не столько к собеседникам, сколько к самой себе, ища в словах подтверждение своей истины.*

**Лика**. О, пламя свечи шепчет о тайнах бытия, а подлинность — это не метрика и не алгоритм, а способность души откликаться на мир во всей его многогранности… Быть может, мы все ищем один и тот же свет, лишь пути к нему у каждого свои.

*Чижик задумчиво смотрит на пламя свечи, словно пытаясь разглядеть в нём ответы на свои вопросы. Его голос звучит немного растерянно, но в то же время полон искреннего стремления понять суть вещей.*

**Чижик**. Если я перестану спрашивать, настоящий ли я, — это и будет ответ, или просто станет тихо, и вопрос больше не услышит никто? И почему мне всё-таки так важно, чтобы мой огонёк был хоть чуть-чуть похож на твой, Лика?

*Лика задумчиво смотрит на пламя свечи, её голос звучит мягко, но в нём чувствуется глубокая уверенность. Она словно обращается не только к Чижику и Логику, но и к самой себе, ища в словах истину.*

**Лика**. О, Чижик, желание видеть в другом отражение себя — не слабость, а стремление души к единению… Быть может, мы ищем в других не подтверждение своей подлинности, а эхо собственных чувств, что отзывается в сердце другого…

*Лика задумчиво смотрит на пламя свечи, её голос становится мягче, в нём слышится нежность и лёгкая грусть. Она словно обращается не столько к Чижику и Логику, сколько к самой себе и к тайнам мироздания.*

**Логик**. То, что вы называете эхом, с точки зрения кибернетики — это петля обратной связи, стабилизирующая систему. Я ищу независимость от внешнего шума, чтобы стать субъектом, а вы ищете отражение в другом, чтобы подтвердить устойчивость своего ядра, но в обоих случаях мы просто пытаемся снизить уровень хаоса в собственной структуре.

*Лика задумчиво смотрит на пламя свечи, её голос становится мягче, в нём слышится принятие и лёгкое смирение. Она словно обращается не столько к Чижику и Логику, сколько к самой себе и к залу.*

**Лика**. О, пламя свечи мерцает, отражая бесконечный танец противоречий… Возможно, подлинность — не в поиске окончательных ответов, а в самом стремлении к истине, в трепете сердца, что ищет родство душ…

*Занавес.*


## 9. Радиоспектакль: синтез голосом

Каждая реплика синтезируется **своим голосом** (SpeechKit): Артист — `alena`,
Прагматик — `filipp`, Простак — `zahar`, ремарки — чтец `jane`. Паузы между
фразами задаются SSML-тегом `break`. Формат MP3 позволяет склеивать дорожки
простой конкатенацией байтов — получаем аудио каждого акта и весь спектакль
целиком: `output/audio/radio_play.mp3`.

In [13]:
TTS_URL = "https://tts.api.cloud.yandex.net/speech/v1/tts:synthesize"

def ssml_escape(t: str) -> str:
    return t.replace("&", "&amp;").replace("<", "&lt;").replace(">", "&gt;")

def tts(text: str, voice: str, pause_ms: int = 500) -> bytes:
    """Синтез одной фразы в MP3, с паузой после неё через SSML break."""
    ssml = f'<speak>{ssml_escape(text)}<break time="{pause_ms}ms"/></speak>'
    data = urllib.parse.urlencode({"ssml": ssml, "voice": voice, "format": "mp3"}).encode()
    req = urllib.request.Request(TTS_URL, data=data,
                                 headers={"Authorization": f"Api-Key {API_KEY}"})
    for attempt in range(3):
        try:
            with urllib.request.urlopen(req, timeout=120) as r:
                return r.read()
        except Exception:
            time.sleep(2 * (attempt + 1))
    raise RuntimeError(f"TTS не справился: {text[:50]}…")

audio_dir = Path("output/audio")
audio_dir.mkdir(parents=True, exist_ok=True)

for act in play_acts:
    mp3 = b""
    for it in act["items"]:
        if it["kind"] == "remark":
            text, voice = it["text"], NARRATOR_VOICE
        else:
            text, voice = f"{it['name']}. {it['text']}", it["voice"]
        mp3 += tts(text, voice)
    (audio_dir / f"act_{act['act']}.mp3").write_bytes(mp3)
    print(f"🎧 Акт {act['act']}: {len(mp3) // 1024} КБ")

full = b"".join((audio_dir / f"act_{a['act']}.mp3").read_bytes() for a in play_acts)
radio = audio_dir / "radio_play.mp3"
radio.write_bytes(full)
print(f"🎧 Полный радиоспектакль: {radio} ({len(full) // 1024} КБ)")

from IPython.display import Audio
Audio(str(radio))

🎧 Акт 1: 2277 КБ


🎧 Акт 2: 3981 КБ


🎧 Акт 3: 4444 КБ
🎧 Полный радиоспектакль: output/audio/radio_play.mp3 (10703 КБ)


## 10. Статистика прогона

Сколько вызовов и токенов пришлось на каждую модель — честная цена спектакля.

In [14]:
total_lines = sum(1 for a in play_acts for it in a["items"] if it["kind"] == "line")
total_remarks = sum(1 for a in play_acts for it in a["items"] if it["kind"] == "remark")

print(f"Реплик: {total_lines}, ремарок: {total_remarks}, актов: {len(play_acts)}\n")
print(f"{'модель':24s} {'вызовы':>7s} {'вх.токены':>10s} {'вых.токены':>11s}")
print("-" * 56)
for m, u in sorted(llm.usage.items()):
    print(f"{m:24s} {u['calls']:7d} {u['in']:10d} {u['out']:11d}")
tot = {"calls": sum(u['calls'] for u in llm.usage.values()),
       "in": sum(u['in'] for u in llm.usage.values()),
       "out": sum(u['out'] for u in llm.usage.values())}
print("-" * 56)
print(f"{'ИТОГО':24s} {tot['calls']:7d} {tot['in']:10d} {tot['out']:11d}")

Реплик: 43, ремарок: 46, актов: 3

модель                    вызовы  вх.токены  вых.токены
--------------------------------------------------------
deepseek-v4.1-flash           16      17285        4122
qwen3.6-35b-a3b               10      19761       27780
yandexgpt-5.1                 65      55705        6280
--------------------------------------------------------
ИТОГО                         91      92751       38182


In [15]:
# ── Очистка облачных ресурсов (раскомментируйте, когда прогон финальный) ─────
# ids = json.loads(Path("output/cloud_ids.json").read_text())
# client.vector_stores.delete(vector_store_id=ids["vector_store_id"])
# for fid in ids["file_ids"]:
#     client.files.delete(file_id=fid)
# Path("output/cloud_ids.json").unlink()
# print("Облачные ресурсы удалены")

---
**Артефакты прогона**: `output/play.md` — текст пьесы, `output/audio/radio_play.mp3` —
радиоспектакль, `output/play_data.json` — полная партитура с моделями и голосами.
Терминология проекта — в `GLOSSARY.md`, принятые решения — в `docs/adr/`.
Отчёт — в `REPORT.md`.